# Missing modality simulation
Standalone script for missing-modality simulation experiments.

In [36]:
import torch
import torch.nn as nn
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from torch.utils.data import TensorDataset, DataLoader
import json
from fusion_model import CrossAttentionFusion

In [ ]:
ROOT = Path("missing_modality_sim.py").resolve().parent.parent
CKPT_DIR = ROOT / "checkpoints"

import joblib

scaler_text         = joblib.load(CKPT_DIR / "scaler_text.joblib")
scaler_audio         = joblib.load(CKPT_DIR / "scaler_audio.joblib")
scaler_video         = joblib.load(CKPT_DIR / "scaler_video.joblib")
speaker_stats        = joblib.load(CKPT_DIR / "speaker_stats_audio.joblib")
speaker_stats_video  = joblib.load(CKPT_DIR / "speaker_stats_video.joblib")

# Audio per-speaker normalization is applied AFTER global StandardScaler, using train-fitted per-speaker stats.
def apply_speaker_norm_audio(X_audio, speaker_ids, stats):
    X_out = X_audio.copy()
    for i, speaker in enumerate(speaker_ids):
        if speaker in stats:
            X_out[i] = (X_audio[i] - stats[speaker]["mean"]) / (stats[speaker]["std"] + 1e-6)
    return X_out

# Video per-speaker normalization is applied AFTER global StandardScaler, using train-fitted per-speaker stats.
def apply_speaker_norm_video(X_video, speaker_ids, stats):
    X_out = X_video.copy()
    for i, speaker in enumerate(speaker_ids):
        if speaker in stats and not np.all(X_video[i] == 0):
            X_out[i] = (X_video[i] - stats[speaker]["mean"]) / (stats[speaker]["std"] + 1e-6)
    return X_out

def load_trained_anchor(anchor, fold_session, ckpt_dir=CKPT_DIR, device="cpu"):
    with open(Path(ckpt_dir) / "cross_attn_config.json") as f:
        cfg = json.load(f)
    model = CrossAttentionFusion(
        cfg["t_in"], cfg["a_in"], cfg["v_in"],
        embed_dim=cfg["embed_dim"], num_heads=cfg["num_heads"],
        dropout=cfg["dropout"], anchor=anchor,
    )
    state = torch.load(
        Path(ckpt_dir) / f"cross_attn_{anchor}anchor_fold{fold_session}.pt",
        map_location=device,
    )
    model.load_state_dict(state)
    model.to(device).eval()
    return model

In [ ]:
# Zero out `modality_to_drop`'s feature vector for a `rate` fraction
def corrupt_modality(X_text, X_audio, X_video, modality_to_drop, rate, rng):
    n = X_text.shape[0]
    drop_mask = rng.random(n) < rate

    Xt, Xa, Xv = X_text.copy(), X_audio.copy(), X_video.copy()
    if modality_to_drop == "text":
        Xt[drop_mask] = 0.0
    elif modality_to_drop == "audio":
        Xa[drop_mask] = 0.0
    elif modality_to_drop == "video":
        Xv[drop_mask] = 0.0
    else:
        raise ValueError(modality_to_drop)
    return Xt, Xa, Xv, drop_mask

Paths

In [39]:
DATASET_ROOT = ROOT / "datasets" / "IEMOCAP_full_release"
FEATURE_DIR = ROOT / "cached_features"
TEXT_FEATURE_DIR = FEATURE_DIR / "text"
VISUAL_FEATURE_DIR = ROOT / "cached_visual_features"
AUDIO_METHOD = "wav2vec2" 

np.random.seed(42)

In [40]:
def load_emotions(eval_path):
    id_to_emotion = {}
    with open(eval_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("["):
                parts = line.split("\t")
                if len(parts) >= 3:
                    utt_id = parts[1]
                    raw_label = parts[2]
                    if raw_label in ["ang", "hap", "sad", "neu"]:
                        id_to_emotion[utt_id.strip()] = raw_label.strip()
                    if raw_label in ["exc"]:
                        id_to_emotion[utt_id.strip()] = "hap"
    return id_to_emotion


def get_speaker_from_utterance(utterance_id):
    sess_match = re.match(r'Ses(\d+)', utterance_id)
    if not sess_match:
        return None
    sess_num = sess_match.group(1)
    parts = utterance_id.split("_")
    if len(parts) >= 3 and parts[-1][0] in ("F", "M"):
        return f"Ses{sess_num}{parts[-1][0]}"
    if "_F" in utterance_id:
        return f"Ses{sess_num}F"
    elif "_M" in utterance_id:
        return f"Ses{sess_num}M"
    return None

Rebuild df_all from cached features + label files

In [ ]:
# Scans TEXT_FEATURE_DIR for cached utt_ids (a directory listing — fast) and cross-references emotion labels from the original EmoEvaluation .txt files (text parsing only, no audio/video decode). Skips any utterance whose audio/visual .npy hasn't been cached yet.
def build_df_from_cache():
    id_to_emotion = {}
    for session_num in range(1, 6):
        eval_dir = DATASET_ROOT / f"Session{session_num}" / "dialog" / "EmoEvaluation"
        if not eval_dir.exists():
            continue
        for eval_path in eval_dir.glob("*.txt"):
            id_to_emotion.update(load_emotions(eval_path))

    rows = []
    for text_path in TEXT_FEATURE_DIR.glob("*_text.npy"):
        utt_id = text_path.stem.replace("_text", "")
        audio_path = FEATURE_DIR / f"{utt_id}_audio_{AUDIO_METHOD}.npy"
        visual_path = VISUAL_FEATURE_DIR / f"{utt_id}_visual.npy"
        emotion = id_to_emotion.get(utt_id)

        if emotion is None or not audio_path.exists() or not visual_path.exists():
            continue

        session_match = re.match(r'Ses(\d+)', utt_id)
        rows.append({
            "utterance_id": utt_id,
            "session": int(session_match.group(1)) if session_match else None,
            "speaker": get_speaker_from_utterance(utt_id),
            "emotion": emotion,
            "text_feature_path": text_path,
            "audio_feature_path": audio_path,
            "visual_feature_path": visual_path,
        })

    df = pd.DataFrame(rows)
    print(f"Loaded {len(df)} cached utterances across "
          f"{df['session'].nunique()} sessions.")
    return df


def load_feature_matrices(df):
    X_text = np.vstack([np.load(p) for p in df["text_feature_path"]])
    X_audio = np.vstack([np.load(p) for p in df["audio_feature_path"]])
    X_video = np.vstack([np.load(p) for p in df["visual_feature_path"]])

    # Global normalization using train-fitted StandardScaler
    X_text  = scaler_text.transform(X_text)
    X_audio = scaler_audio.transform(X_audio)
    X_video = scaler_video.transform(X_video)

    # Per-speaker normalization using train-fitted per-speaker stats
    speaker_ids = df["speaker"].values
    X_audio = apply_speaker_norm_audio(X_audio, speaker_ids, speaker_stats)
    X_video = apply_speaker_norm_video(X_video, speaker_ids, speaker_stats_video)

    return X_text, X_audio, X_video

Missing-modality simulation

In [ ]:
SEVERITY_BANDS = {
    "mild":     (0.0, 0.30),
    "moderate": (0.30, 0.70),
    "severe":   (0.70, 1.00),
}

# Draws a random missingness rate from the specified severity band.
def sample_missing_rate(band, rng):
    lo, hi = SEVERITY_BANDS[band]
    return rng.uniform(lo, hi)

# Simulates missingness by zeroing out (or NaN-masking) a fraction `rate` of samples for the chosen modality.
def drop_modality(X_text, X_audio, X_video, modality, rate, rng, mode="zero"):
    n = X_text.shape[0]
    n_drop = int(round(rate * n))
    drop_idx = rng.choice(n, size=n_drop, replace=False)
    missing_mask = np.zeros(n, dtype=bool)
    missing_mask[drop_idx] = True

    X_text_out, X_audio_out, X_video_out = X_text.copy(), X_audio.copy(), X_video.copy()
    fill_value = np.nan if mode == "nan" else 0.0

    if modality == "text":
        X_text_out[drop_idx] = fill_value
    elif modality == "audio":
        X_audio_out[drop_idx] = fill_value
    elif modality == "video":
        X_video_out[drop_idx] = fill_value
    else:
        raise ValueError(f"Unknown modality: {modality}")

    return X_text_out, X_audio_out, X_video_out, missing_mask

Recovery baselines: zero-imputation and mean-imputation

In [ ]:
# Computes the per-feature mean to use for mean-imputation.
def compute_mean_fill_vector(X_train, missing_mask_train=None):
    if missing_mask_train is not None:
        available = X_train[~missing_mask_train]
        if len(available) == 0:
            raise ValueError("No non-missing training rows to compute "
                              "a mean from — check your missing rate.")
        return available.mean(axis=0)
    return X_train.mean(axis=0)

# Fills in rows flagged as missing in `missing_mask` using a recovery strategy. Call this AFTER drop_modality(..., mode="nan") so the same missingness mask can be reused across strategies for a fair, paired comparison (same dropped rows, different fill).
def impute_missing(X, missing_mask, strategy="zero", fill_vector=None):
    X_imputed = X.copy()

    if strategy == "zero":
        X_imputed[missing_mask] = 0.0

    elif strategy == "mean":
        if fill_vector is None:
            raise ValueError("mean imputation requires fill_vector "
                              "(see compute_mean_fill_vector)")
        X_imputed[missing_mask] = fill_vector

    else:
        raise ValueError(f"Unknown imputation strategy: {strategy}")

    return X_imputed

Non-recovery baselines: attention-masking and gated cross-attention

In [ ]:
# Simple non-recovery baseline: text-anchored cross-attention where a missing modality is HARD-EXCLUDED from attention via key_padding_mask, rather than learned-down-weighted. If a modality is missing for a given sample, the model attends to it with zero weight — full stop.
class AttentionMaskingFusion(nn.Module):
    def __init__(self, text_dim, audio_dim, video_dim, hidden_dim=256,
                 n_classes=4, n_heads=4):
        super().__init__()
        self.proj_t = nn.Linear(text_dim, hidden_dim)
        self.proj_a = nn.Linear(audio_dim, hidden_dim)
        self.proj_v = nn.Linear(video_dim, hidden_dim)

        self.attn_audio = nn.MultiheadAttention(hidden_dim, n_heads, batch_first=True)
        self.attn_video = nn.MultiheadAttention(hidden_dim, n_heads, batch_first=True)
        self.ln_audio = nn.LayerNorm(hidden_dim)
        self.ln_video = nn.LayerNorm(hidden_dim)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, text, audio, video, audio_present, video_present):
        t = self.proj_t(text).unsqueeze(1)  
        a = self.proj_a(audio).unsqueeze(1) 
        v = self.proj_v(video).unsqueeze(1)

        # key_padding_mask: True = IGNORE this position.
        # Shape (B, 1) since each modality contributes a single token here.
        audio_pad_mask = (~audio_present).unsqueeze(1)
        video_pad_mask = (~video_present).unsqueeze(1)

        attn_a, _ = self.attn_audio(t, a, a, key_padding_mask=audio_pad_mask)
        attn_v, _ = self.attn_video(t, v, v, key_padding_mask=video_pad_mask)

        attn_a = torch.nan_to_num(attn_a, nan=0.0)
        attn_v = torch.nan_to_num(attn_v, nan=0.0)

        ctx_a = self.ln_audio(attn_a + t).squeeze(1)
        ctx_v = self.ln_video(attn_v + t).squeeze(1)
        t_s = t.squeeze(1)

        fused = torch.cat([t_s, ctx_a, ctx_v], dim=1)
        return self.classifier(fused)

In [ ]:
# Minimal classifier paired with the recovery baselines (zero/mean imputation). No cross-attention — just concatenates the (already-imputed) modality vectors through an MLP. Kept deliberately simple so any F1 difference between zero- and mean-imputation is attributable to the imputation strategy, not the fusion architecture.
class EarlyFusionClassifier(nn.Module):
    def __init__(self, text_dim, audio_dim, video_dim, hidden_dim=256, n_classes=4):
        super().__init__()
        in_dim = text_dim + audio_dim + video_dim
        self.classifier = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, text, audio, video, audio_present=None, video_present=None):
        # presence flags accepted-but-unused so this shares a call
        # signature with the non-recovery models in the sweep loop below
        fused = torch.cat([text, audio, video], dim=1)
        return self.classifier(fused)

In [46]:
def get_loso_split(df_all, test_session):
    train_df = df_all[df_all["session"] != test_session].reset_index(drop=True)
    test_df = df_all[df_all["session"] == test_session].reset_index(drop=True)
    return train_df, test_df


def train_classifier(model, X_text, X_audio, X_video, audio_present, video_present,
                      y, device, n_epochs=15, lr=1e-3, batch_size=32):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    dataset = TensorDataset(
        torch.tensor(X_text, dtype=torch.float32),
        torch.tensor(X_audio, dtype=torch.float32),
        torch.tensor(X_video, dtype=torch.float32),
        torch.tensor(audio_present, dtype=torch.bool),
        torch.tensor(video_present, dtype=torch.bool),
        torch.tensor(y, dtype=torch.long),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.0
        for t, a, v, ap, vp, yb in loader:
            t, a, v, ap, vp, yb = [x.to(device) for x in (t, a, v, ap, vp, yb)]
            optimizer.zero_grad()
            out = model(t, a, v, ap, vp)
            logits = out[0] if isinstance(out, tuple) else out
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    return model


def evaluate_classifier(model, X_text, X_audio, X_video, audio_present, video_present,
                         y, device):
    model.eval()
    with torch.no_grad():
        t = torch.tensor(X_text, dtype=torch.float32).to(device)
        a = torch.tensor(X_audio, dtype=torch.float32).to(device)
        v = torch.tensor(X_video, dtype=torch.float32).to(device)
        ap = torch.tensor(audio_present, dtype=torch.bool).to(device)
        vp = torch.tensor(video_present, dtype=torch.bool).to(device)
        out = model(t, a, v, ap, vp)
        logits = out[0] if isinstance(out, tuple) else out
        preds = logits.argmax(dim=1).cpu().numpy()
    return f1_score(y, preds, average="weighted")

In [ ]:
# Logs the gated cross-attention model's gate weights, split by whether the audio modality was present or missing for each sample. Watches for the "dominance" failure mode flagged in the missing-modality literature: if the gate collapses to relying almost entirely on the anchor (text) modality whenever audio is missing — rather than gracefully redistributing weight between text and the modality that IS still present (video) — that's a sign the gate isn't actually reasoning about missingness, just defaulting to the anchor.
def analyze_gate_dominance(model, X_text, X_audio, X_video,
                            audio_present, video_present, device):
    model.eval()
    with torch.no_grad():
        t = torch.tensor(X_text, dtype=torch.float32).to(device)
        a = torch.tensor(X_audio, dtype=torch.float32).to(device)
        v = torch.tensor(X_video, dtype=torch.float32).to(device)
        ap = torch.tensor(audio_present, dtype=torch.bool).to(device)
        vp = torch.tensor(video_present, dtype=torch.bool).to(device)

        _, gate_w = model(t, a, v, ap, vp)
        gate_w = gate_w.cpu().numpy()

    present_mask = audio_present
    missing_mask = ~audio_present

    stats = {
        "n_present": int(present_mask.sum()),
        "n_missing": int(missing_mask.sum()),
    }

    component_names = ["text_anchor", "audio_ctx", "video_ctx"]
    for i, name in enumerate(component_names):
        stats[f"{name}_mean_when_present"] = (
            float(gate_w[present_mask, i].mean()) if present_mask.any() else float("nan")
        )
        stats[f"{name}_mean_when_missing"] = (
            float(gate_w[missing_mask, i].mean()) if missing_mask.any() else float("nan")
        )

    return stats

In [ ]:
def run_severity_sweep(df_all, le, test_sessions, modality_to_drop="audio",
                        n_epochs=15, seed=42):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []
    gate_dominance_records = []

    for test_session in test_sessions:
        print(f"\n{'='*60}\nLOSO fold — test session {test_session}\n{'='*60}")
        train_df, test_df = get_loso_split(df_all, test_session)

        X_text_tr, X_audio_tr, X_video_tr = load_feature_matrices(train_df)
        X_text_te, X_audio_te, X_video_te = load_feature_matrices(test_df)

        y_tr = le.transform(train_df["emotion"])
        y_te = le.transform(test_df["emotion"])

        n_classes = len(le.classes_)
        text_dim, audio_dim, video_dim = (
            X_text_tr.shape[1], X_audio_tr.shape[1], X_video_tr.shape[1]
        )

        full_audio_present_tr = np.ones(len(train_df), dtype=bool)
        full_video_present_tr = np.ones(len(train_df), dtype=bool)

        for band in SEVERITY_BANDS:
            rng = np.random.default_rng(seed)
            rate = sample_missing_rate(band, rng)

            X_text_nan, X_audio_nan, X_video_nan, mask_te = drop_modality(
                X_text_te, X_audio_te, X_video_te,
                modality=modality_to_drop, rate=rate, rng=rng, mode="nan"
            )
            audio_present_te = (~mask_te if modality_to_drop == "audio"
                                 else np.ones(len(test_df), dtype=bool))
            video_present_te = (~mask_te if modality_to_drop == "video"
                                 else np.ones(len(test_df), dtype=bool))

            # Recovery baselines: zero- and mean-imputation, followed by a simple MLP classifier. Trained fresh for each severity band, since the missingness pattern is different each time.
            for strategy in ["zero", "mean"]:
                X_text_fill, X_audio_fill, X_video_fill = X_text_te, X_audio_te, X_video_te

                if modality_to_drop == "text":
                    fill_vec = (np.zeros(text_dim) if strategy == "zero"
                                else compute_mean_fill_vector(X_text_tr))
                    X_text_fill = impute_missing(X_text_nan, mask_te, strategy, fill_vec)
                elif modality_to_drop == "audio":
                    fill_vec = (np.zeros(audio_dim) if strategy == "zero"
                                else compute_mean_fill_vector(X_audio_tr))
                    X_audio_fill = impute_missing(X_audio_nan, mask_te, strategy, fill_vec)
                elif modality_to_drop == "video":
                    fill_vec = (np.zeros(video_dim) if strategy == "zero"
                                else compute_mean_fill_vector(X_video_tr))
                    X_video_fill = impute_missing(X_video_nan, mask_te, strategy, fill_vec)

                model = EarlyFusionClassifier(text_dim, audio_dim, video_dim, n_classes=n_classes)
                model = train_classifier(model, X_text_tr, X_audio_tr, X_video_tr,
                                          full_audio_present_tr, full_video_present_tr,
                                          y_tr, device, n_epochs=n_epochs)
                f1 = evaluate_classifier(model, X_text_fill, X_audio_fill, X_video_fill,
                                          audio_present_te, video_present_te, y_te, device)
                results.append({
                    "test_session": test_session, "band": band, "rate": rate,
                    "method": f"recovery_{strategy}", "modality_dropped": modality_to_drop,
                    "weighted_f1": f1,
                })
                print(f"[{band:>8}] recovery_{strategy:<5} F1={f1:.4f}")

            # Non-recovery baseline: text-anchored cross-attention with hard masking of the missing modality. Trained fresh for each severity band, since the missingness pattern is different each time.
            X_text_zero_te = (impute_missing(X_text_nan, mask_te, "zero")
                    if modality_to_drop == "text" else X_text_te)
            X_audio_zero_te = (impute_missing(X_audio_nan, mask_te, "zero")
                                if modality_to_drop == "audio" else X_audio_te)
            X_video_zero_te = (impute_missing(X_video_nan, mask_te, "zero")
                                if modality_to_drop == "video" else X_video_te)

            model = AttentionMaskingFusion(text_dim, audio_dim, video_dim, n_classes=n_classes)
            model = train_classifier(model, X_text_tr, X_audio_tr, X_video_tr,
                                      full_audio_present_tr, full_video_present_tr,
                                      y_tr, device, n_epochs=n_epochs)
            f1 = evaluate_classifier(model, X_text_zero_te, X_audio_zero_te, X_video_zero_te,
                          audio_present_te, video_present_te, y_te, device)
            results.append({
                "test_session": test_session, "band": band, "rate": rate,
                "method": "mask", "modality_dropped": modality_to_drop,
                "weighted_f1": f1,
            })
            print(f"[{band:>8}] mask         F1={f1:.4f}")

            # Actual missing-modality model: text-anchored cross-attention with learned gating. Trained fresh for each severity band, since the missingness pattern is different each time.
            for anchor in ("text", "audio", "video", "avg"):
                anchor_model = load_trained_anchor(anchor, fold_session=test_session, device=device)
                with torch.no_grad():
                    t = torch.tensor(X_text_zero_te, dtype=torch.float32).to(device)
                    a = torch.tensor(X_audio_zero_te, dtype=torch.float32).to(device)
                    v = torch.tensor(X_video_zero_te, dtype=torch.float32).to(device)
                    logits, gate_weights, _, _ = anchor_model(t, a, v, return_attn=True)
                    preds = logits.argmax(dim=1).cpu().numpy()

                f1 = f1_score(y_te, preds, average="weighted")
                results.append({
                    "test_session": test_session, "band": band, "rate": rate,
                    "method": f"gate_{anchor}anchor", "modality_dropped": modality_to_drop,
                    "weighted_f1": f1,
                })
                print(f"[{band:>8}] gate_{anchor:<7} F1={f1:.4f}")

                # Gate dominance analysis: log the mean gate weights for the anchor modality when the dropped modality is present vs. missing, to watch for the "dominance" failure mode flagged in the missing-modality literature.
                gw = gate_weights.detach().cpu().numpy()
                present_mask = ~mask_te
                missing_mask = mask_te
                dominance_stats = {
                    "test_session": test_session, "band": band, "rate": rate,
                    "anchor": anchor, "modality_dropped": modality_to_drop,
                    "text_anchor_mean_when_present": float(gw[present_mask, 0].mean()) if present_mask.any() else float("nan"),
                    "text_anchor_mean_when_missing": float(gw[missing_mask, 0].mean()) if missing_mask.any() else float("nan"),
                }
                gate_dominance_records.append(dominance_stats)

    return pd.DataFrame(results), pd.DataFrame(gate_dominance_records)

Smoke test on a small subset

In [49]:
df_all = build_df_from_cache()

le = LabelEncoder()
le.fit(df_all["emotion"])
print(f"Emotion classes: {list(le.classes_)}")

TEST_SESSIONS_TO_RUN = [1, 2, 3, 4, 5]

available_sessions = set(df_all["session"].dropna().unique())
missing_requested = [s for s in TEST_SESSIONS_TO_RUN if s not in available_sessions]
if missing_requested:
    raise ValueError(
        f"Requested test session(s) {missing_requested} not found in "
        f"cached data. Available sessions: {sorted(available_sessions)}. "
        f"Check whether text.py's extraction loop has processed these "
        f"sessions yet."
    )

all_results, all_dominance = [], []
for modality_to_drop in ("text", "audio", "video"):
    print(f"\n{'#'*60}\nSTRESS TEST — dropping modality: {modality_to_drop}\n{'#'*60}")
    r_df, d_df = run_severity_sweep(
        df_all, le, test_sessions=TEST_SESSIONS_TO_RUN,
        modality_to_drop=modality_to_drop, n_epochs=15,
    )
    all_results.append(r_df)
    all_dominance.append(d_df)

results_df = pd.concat(all_results, ignore_index=True)
gate_dominance_df = pd.concat(all_dominance, ignore_index=True)

Loaded 5531 cached utterances across 5 sessions.
Emotion classes: ['ang', 'hap', 'neu', 'sad']

############################################################
STRESS TEST — dropping modality: text
############################################################

LOSO fold — test session 1
[    mild] recovery_zero  F1=0.6518
[    mild] recovery_mean  F1=0.6399
[    mild] mask         F1=0.6237
[    mild] gate_text    F1=0.6254
[    mild] gate_audio   F1=0.6277
[    mild] gate_video   F1=0.6337
[    mild] gate_avg     F1=0.6468
[moderate] recovery_zero  F1=0.6099
[moderate] recovery_mean  F1=0.5633
[moderate] mask         F1=0.5621
[moderate] gate_text    F1=0.5874
[moderate] gate_audio   F1=0.6026
[moderate] gate_video   F1=0.6015
[moderate] gate_avg     F1=0.6126
[  severe] recovery_zero  F1=0.5702
[  severe] recovery_mean  F1=0.5594
[  severe] mask         F1=0.5793
[  severe] gate_text    F1=0.5484
[  severe] gate_audio   F1=0.5851
[  severe] gate_video   F1=0.5860
[  severe] gate_avg     

Aggregate

In [ ]:
summary = (
    results_df
    .groupby(["modality_dropped", "method", "band"])["weighted_f1"]
    .agg(mean_f1="mean", variance_f1="var", n_folds="count")
    .reset_index()
)

table5_df = (
    results_df
    .groupby(["modality_dropped", "method", "band"])["weighted_f1"]
    .agg(mean_f1="mean", std_f1=lambda x: x.std(), n_folds="count")
    .reset_index()
    .round(4)
)
print(table5_df.to_markdown(index=False))
table5_df.to_csv("table5_missing_modality_sweep.csv", index=False)

# One pivot table per dropped modality, for easier reading in the paper.
for modality in ("text", "audio", "video"):
    sub = table5_df[table5_df["modality_dropped"] == modality]
    pivot = sub.pivot(index="method", columns="band", values="mean_f1").round(4)
    print(f"\n=== {modality} dropped ===")
    print(pivot.to_markdown())
    pivot.to_csv(f"table5_missing_modality_pivot_{modality}.csv")

| modality_dropped   | method           | band     |   mean_f1 |   std_f1 |   n_folds |
|:-------------------|:-----------------|:---------|----------:|---------:|----------:|
| audio              | gate_audioanchor | mild     |    0.6068 |   0.0306 |         5 |
| audio              | gate_audioanchor | moderate |    0.5443 |   0.0313 |         5 |
| audio              | gate_audioanchor | severe   |    0.4902 |   0.0276 |         5 |
| audio              | gate_avganchor   | mild     |    0.6453 |   0.0277 |         5 |
| audio              | gate_avganchor   | moderate |    0.5909 |   0.0252 |         5 |
| audio              | gate_avganchor   | severe   |    0.543  |   0.032  |         5 |
| audio              | gate_textanchor  | mild     |    0.6154 |   0.0165 |         5 |
| audio              | gate_textanchor  | moderate |    0.5785 |   0.0136 |         5 |
| audio              | gate_textanchor  | severe   |    0.5447 |   0.0182 |         5 |
| audio              | gate_vide

In [51]:
for modality in ("text", "audio", "video"):
    sub = gate_dominance_df[gate_dominance_df["modality_dropped"] == modality]
    if sub.empty:
        continue
    print(f"\n{'='*60}\nGate-weight dominance check ({modality} dropped)\n{'='*60}")
    dominance_summary = sub.groupby(["anchor", "band"])[
        ["text_anchor_mean_when_present", "text_anchor_mean_when_missing"]
    ].mean()
    dominance_summary["text_anchor_shift"] = (
        dominance_summary["text_anchor_mean_when_missing"]
        - dominance_summary["text_anchor_mean_when_present"]
    )
    print(dominance_summary.round(4))
    dominance_summary.to_csv(f"table6_gate_dominance_{modality}dropped.csv")


Gate-weight dominance check (text dropped)
                 text_anchor_mean_when_present  text_anchor_mean_when_missing  \
anchor band                                                                     
audio  mild                             0.0277                         0.0276   
       moderate                         0.0270                         0.0282   
       severe                           0.0386                         0.0271   
avg    mild                             0.0070                         0.0018   
       moderate                         0.0076                         0.0022   
       severe                           0.0038                         0.0025   
text   mild                             0.0082                         0.0007   
       moderate                         0.0089                         0.0007   
       severe                           0.0088                         0.0007   
video  mild                             0.0011                   